# PohaRana / KabulLearn — Complete Project Documentation

**Platform:** KabulLearn (`kabullearn.com`) — multilingual online learning platform for Afghan learners worldwide.  
**Codebase name:** PohaRana (Pashto: "Education")  
**Status:** Live / Production  
**Last updated:** June 2026

---

## Quick Reference

| Item | Value |
|---|---|
| Production URL | kabullearn.com |
| Dev server | `rm -rf .next && npm run dev -- --port 3002` |
| Database | PostgreSQL on Neon (project ID: `snowy-band-10045491`) |
| Auth | Auth.js v5 (next-auth@beta), JWT strategy |
| Payments | Stripe (Checkout Sessions + webhooks) |
| AI | OpenAI (course chat, classification, translation) |
| Storage | Vercel Blob (avatars, uploads) |
| Rate-limiting | Upstash Redis |
| Deployment | Vercel (kabullearn.com), Cloudflare (kabulhub.com) |
| Type safety | TypeScript 5, Zod 4, Prisma 6 |

## 1. Tech Stack

### Framework & Runtime

| Layer | Technology | Version |
|---|---|---|
| Framework | Next.js (App Router) | ^16.2.9 |
| Runtime | React | 19.2.6 |
| Language | TypeScript | ^5.8.3 |
| Styling | Tailwind CSS v4 | 4.1.8 |
| Node | Node.js (serverless via Vercel) | LTS |

### Backend / Data

| Layer | Technology | Notes |
|---|---|---|
| ORM | Prisma | v6; client generated on `postinstall` and `build` |
| Database | PostgreSQL | Hosted on Neon; pooled via `DATABASE_URL`, direct via `DATABASE_URL_UNPOOLED` |
| Auth | Auth.js (next-auth v5 beta) | JWT strategy; PrismaAdapter for full Node flow |
| Validation | Zod v4 | All Server Action inputs validated before DB |

### Integrations

| Service | Purpose | SDK |
|---|---|---|
| Stripe | Course payments, promo codes, webhooks | `stripe` ^22 |
| OpenAI | Course chatbot, course classification, PS/FA translation | `openai` ^6 |
| Vercel Blob | Avatar uploads, file storage | `@vercel/blob` ^2 |
| Upstash Redis | Rate-limiting, counters | `@upstash/redis` ^1 |
| Neon | PostgreSQL hosting (serverless) | Direct via Prisma `DATABASE_URL` |
| Nodemailer | Transactional email (verification, password reset) | Built into lib |

### Client-side packages

| Package | Purpose |
|---|---|
| `canvas-confetti` | Purchase celebration confetti effect |
| `qrcode` | QR code generation for certificate verification (client-side, no third party) |
| `pdfkit` | Server-side PDF generation for certificate download |
| `bcryptjs` | Password hashing (Node-only, never imported in edge code) |
| `iconv-lite` | Character encoding for PDF text |

## 2. Architecture Overview

### Role Separation (strict, no overlap)

Three roles with completely separate route namespaces. Middleware enforces boundaries at the edge — no DB lookup.

| Role | Primary Routes | How Created |
|---|---|---|
| `STUDENT` | `/`, `/courses/*`, `/dashboard/*` | Self-registers at `/register` |
| `EDUCATOR` | `/educator/*` | Admin promotes from STUDENT via `/admin` |
| `ADMIN` | `/admin/*` | Created only via `npm run db:seed` |

After login, `app/auth/redirect/page.tsx` reads the JWT role and redirects each role to its home page.

### Auth Split (edge vs. Node)

Auth.js is deliberately configured in two files:

- **`auth.config.ts`** — edge-safe, zero Node-only imports (no Prisma, no bcrypt); used by `proxy.ts` middleware
- **`auth.ts`** — full Node config with `PrismaAdapter` and `bcryptjs`; used by all Server Components and Actions

JWT tokens embed: `role`, `status`, `sessionVersion`. This means session validation never touches the DB — only initial login does.

### App Router Rendering Pattern

```
Server Component (async page)
  ├── calls auth() / DB queries / Server Actions (server-side only)
  └── passes data down to Client Components ("use client")
      ├── handles interactivity, state, forms
      └── calls Server Actions for writes (never raw fetch to own API)
```

- **Server Actions** (`"use server"`) handle all writes: enrollment, quiz submission, progress updates, etc.
- **API Routes** (`app/api/*`) are used only for: Stripe webhooks, heartbeat tracking, file uploads, OpenAI streaming, and OAuth callbacks.
- The pattern `ActionResult<T> = { ok: true; data: T } | { ok: false; error: string }` is used universally — Server Actions never throw to clients.

### Module Gating (dual-source)

Progress unlock uses two sources merged on the client:

1. **Server**: `UserProgress` records where `lesson.isFinalTest = true` and `status = COMPLETED`
2. **Client**: `localStorage` key `poharana:{courseId}:{moduleId}:quizPassed` for optimistic unlocking

Rule: Module 0 is always unlocked. Module N requires the final quiz of Module N−1 to be `COMPLETED`. Server values always win over localStorage.

## 3. Route Map

### Public Routes
```
/                          Homepage (server, reads locale from cookie)
/courses                   Course catalog
/courses/[courseId]        Course overview (enrollment, purchase)
/courses/[courseId]/lessons/[lessonId]   Lesson viewer (VIDEO / READING)
/courses/[courseId]/quizzes/[lessonId]   Quiz page
/courses/[courseId]/certificate          Certificate viewer
/courses/[courseId]/certificate/download PDF download (route handler)
/courses/[courseId]/thank-you            Post-payment confirmation + confetti
/verify/[code]             Certificate verification lookup
/catalog                   Full searchable catalog
/creators/[username]       Public creator profile
/learning-paths/[slug]     Curated learning path
/register                  Student registration
/login                     Login (credentials + OAuth)
/forgot-login              Password reset request
/reset-password            Password reset (with token)
/verify-request            Email verification pending page
/about  /contact  /privacy /terms /support   Static info pages
/for-educators  /educator-guidelines  /educator-resources   Educator marketing
```

### Authenticated Student Routes (`/dashboard/*`)
```
/dashboard                 Dashboard overview (redirects → my-courses)
/dashboard/my-courses      Enrolled course list
/dashboard/certificates    Earned certificates
/dashboard/messages        DM inbox (student ↔ educator)
/dashboard/settings        Profile, avatar, password, locale
/messages                  Alias → dashboard/messages
/settings                  Alias → dashboard/settings
```

### Educator Routes (`/educator/*`, role=EDUCATOR required)
```
/educator                  Creator dashboard (analytics, course list)
/educator/courses/new      Create new course
/educator/courses/[id]     Course editor (modules, lessons, quiz builder)
/educator/messages         DM inbox (educator ↔ students)
```

### Admin Routes (`/admin/*`, role=ADMIN required)
```
/admin                     Admin overview (users, courses, stats)
/admin/ai                  AI tools (reindex, classify)
/admin/audit               Admin audit log
/admin/messages            Broadcast / direct message composer
```

### API Routes
```
/api/auth/[...nextauth]    Auth.js OAuth + credentials handler
/api/stripe/checkout       Create Stripe checkout session
/api/stripe/webhook        Stripe webhook (marks PAID, upserts enrollment)
/api/lesson/heartbeat      Video progress heartbeat (POST, protected)
/api/chat                  OpenAI streaming chat for course chatbot
/api/chat/feedback         Chat thumbs feedback
/api/enroll                Free enrollment endpoint
/api/promo/validate        Promo code validation (100% = skip Stripe)
/api/search                Full-text course search
/api/upload/avatar         Vercel Blob avatar upload
/api/educator/*            Educator AI tools (classify, translate, profile-context)
/api/og/course             Dynamic OG image for course
/api/cron/reengage         Re-engagement email cron (Vercel cron)
/api/admin/reindex         Trigger embedding re-index (admin-only)
/api/contact-ticket        Public contact form
```

## 4. Middleware (`proxy.ts`)

The middleware file is `proxy.ts` (exported as `middleware` via Next.js convention). It runs at the edge on every matched request.

### What it does

1. **Coming-soon redirect**: If the host is `kabullearn.com` and path is `/`, rewrite to `/coming-soon.html` static file.
2. **Verification gate**: If any protected path is accessed while `status === "VERIFICATION_PENDING"`, redirect to `/verify-request`.
3. **Role enforcement**:
   - `/admin/*` requires `role === "ADMIN"` — others redirected to `/dashboard`
   - `/educator/*` requires `role === "EDUCATOR"` — admins → `/admin`, students → `/dashboard`
   - Unauthenticated users on `/admin` or `/educator` → `/login?callbackUrl=...`
4. **CORS headers for course embeds**: Sets `Permissions-Policy`, `Cross-Origin-Opener-Policy`, `Cross-Origin-Resource-Policy` on `/courses/*` responses.

### Protected paths (matcher config)
```
/  /admin/:path*  /educator/:path*  /dashboard/:path*  /courses/:path*
/api/enroll  /api/promo/:path*  /api/educator/:path*  /api/chat/:path*  /api/lesson/:path*
```

### Key design decision
Middleware uses `auth.config.ts` (edge-safe) — never imports Prisma or bcrypt. JWT decoding happens entirely in the edge runtime from the session cookie with no DB round-trip.

## 5. Database Schema (all models)

**Provider:** PostgreSQL on Neon (serverless). Managed via Prisma 6. Migrations in `prisma/migrations/`.

### Enums

| Enum | Values |
|---|---|
| `UserRole` | `STUDENT`, `EDUCATOR`, `ADMIN` |
| `UserStatus` | `VERIFICATION_PENDING`, `ACTIVE`, `SUSPENDED` |
| `CourseStatus` | `DRAFT`, `PENDING_REVIEW`, `PUBLISHED` |
| `LessonType` | `VIDEO`, `READING`, `QUIZ` |
| `QuestionType` | `SINGLE_CHOICE`, `MULTIPLE_CHOICE`, `TEXT_INPUT` |
| `ProgressStatus` | `NOT_STARTED`, `IN_PROGRESS`, `COMPLETED` |
| `PaymentStatus` | `PENDING`, `PAID`, `FAILED`, `EXPIRED`, `REFUNDED` |
| `PaymentPurpose` | `COURSE`, `DONATION` |
| `DiscountType` | `PERCENT`, `FIXED_CENTS` |
| `ReviewEventType` | `SUBMITTED`, `PUBLISHED`, `RETURNED` |
| `NotificationStatus` | `QUEUED`, `SENT`, `FAILED` |
| `AppNotificationKind` | `NEW_LESSON`, `DISCUSSION_REPLY`, `STREAK_ALERT`, `COURSE_REVIEW`, `GENERAL` |
| `EducatorRequestStatus` | `PENDING`, `APPROVED`, `REJECTED` |

### Identity & Auth Models

| Model | Key Fields | Notes |
|---|---|---|
| `User` | `id` (cuid), `email` (unique), `role`, `status`, `locale`, `sessionVersion` | Central user record; `onDelete: Restrict` for Course.authorId |
| `Account` | OAuth provider + account IDs | `@@unique([provider, providerAccountId])` |
| `Session` | `sessionToken` (unique) | Auth.js managed |
| `VerificationToken` | `identifier`, `token`, `expires` | Email verification |
| `AdminAuditLog` | `actorId`, `action`, `targetType`, `targetId` | Every admin action logged |

### Content Models

| Model | Key Fields | Notes |
|---|---|---|
| `Course` | `id`, `slug` (unique), `status`, `isPaid`, `price`, `currency`, `authorId`, `authorProfileId` | `titleEn`, `titlePs`, `titleFa`; indexes on status, authorId, isPaid |
| `Module` | `courseId`, `order` | `@@unique([courseId, order])` |
| `Lesson` | `moduleId`, `order`, `type`, `isFinalTest`, `passingScore`, `youtubeUrl` | `@@unique([moduleId, order])`; `titleEn/Ps/Fa`, `bodyEn/Ps/Fa` |
| `Quiz` | `lessonId` (unique) | One quiz shell per quiz lesson |
| `Question` | `quizId`, `order`, `type`, `promptEn/Ps/Fa` | `@@unique([quizId, order])` |
| `AnswerChoice` | `questionId`, `order`, `textEn/Ps/Fa`, `isCorrect` | `@@unique([questionId, order])` |
| `CreatorProfile` | `username` (unique), `userId` (unique), `displayName`, `bio`, `avatarUrl` | Public instructor identity separate from auth |
| `CourseInstructor` | `courseId`, `profileId`, `order` | `@@unique([courseId, profileId])`; bridge table |

### Learning Activity Models

| Model | Key Fields | Notes |
|---|---|---|
| `Enrollment` | `userId`, `courseId` | `@@unique([userId, courseId])` |
| `UserProgress` | `userId`, `lessonId`, `status`, `score` | `@@unique([userId, lessonId])`; `COMPLETED` gates next module |
| `LessonHeartbeat` | `userId`, `lessonId`, `courseId`, `consumedPct`, `positionSec` | Video telemetry; indexed `(userId, lessonId, createdAt)` |
| `QuizAttemptSession` | `userId`, `lessonId`, `startedAt`, `expiresAt` | Anti-replay; 3 failed attempts per 8-hour window |
| `QuizSubmission` | `userId`, `lessonId`, `score`, `passed`, `submittedAt` | Indexed `(userId, lessonId, passed, submittedAt)` |
| `QuizSubmissionAnswer` | `submissionId`, `questionId`, `answerChoiceId`, `textAnswer` | Per-question record |

### Payments & Commerce Models

| Model | Key Fields | Notes |
|---|---|---|
| `Payment` | `userId`, `courseId`, `purpose`, `status`, `amountCents`, `currency`, `stripeCheckoutSessionId`, `stripePaymentIntentId`, `promoCodeId` | `PENDING` → Stripe → `PAID` via webhook |
| `PromoCode` | `code` (unique), `discountType`, `discountValue`, `maxUses`, `usedCount`, `isActive`, `expiresAt` | 100% PERCENT discount skips Stripe entirely |

### Credentials & Quality Models

| Model | Key Fields | Notes |
|---|---|---|
| `Certificate` | `userId`, `courseId`, `uuid`, `verificationCode`, `issuedAt`, `grade` | `@@unique([userId, courseId])` |
| `CourseRating` | `userId`, `courseId`, `rating`, `comment` | `@@unique([userId, courseId])` |
| `CourseReviewEvent` | `courseId`, `actorId`, `type`, `notes` | Audit trail: SUBMITTED → PUBLISHED/RETURNED |
| `CourseReviewChecklistItem` | `courseId`, `key`, `checked` | Admin review checklist |

### Community & Notifications

| Model | Key Fields | Notes |
|---|---|---|
| `DiscussionThread` | `courseId`, `authorId`, `title`, `body` | |
| `DiscussionReply` | `threadId`, `authorId`, `body` | |
| `DirectMessage` | `senderId`, `recipientId`, `body`, `readAt` | Student ↔ educator DMs; admin messages are read-only from student side |
| `NotificationLog` | `email`, `type`, `status`, `sentAt` | Email dispatch tracking |
| `AppNotification` | `userId`, `kind`, `title`, `body`, `readAt` | In-app notification bell |

### Other Models

| Model | Purpose |
|---|---|
| `LessonNote` | Student private lesson notes |
| `LessonBookmark` | Student bookmarked lessons |
| `UserStreak` | Daily learning streak tracking |
| `LearningPath` | Curated multi-course learning paths |
| `LearningPathCourse` | Bridge: learning path ↔ course |
| `CourseAnnouncement` | Educator announcements to enrolled students |
| `CourseTag` / `CourseTagRelation` | Course categorization |
| `ContentEmbedding` | Vector embeddings for AI semantic search |
| `AiChatLog` | Chat history per user per course |
| `SiteSetting` | Key-value store for admin-controlled site settings |
| `EducatorRequest` | Student's application to become educator |

## 6. Key Data Pipelines

### 6.1 Registration & Email Verification Pipeline

```
POST /register → register-actions.ts
  1. Zod validate { name, email, password }
  2. Hash password with bcryptjs (10 rounds)
  3. Create User (status: VERIFICATION_PENDING)
  4. Create VerificationToken (24h expiry)
  5. Send verification email via nodemailer (token in link)
  6. User clicks link → /api/auth/verify → mark emailVerified, status: ACTIVE
  7. Redirect → /auth/redirect → role-based home
```

### 6.2 Free Enrollment Pipeline

```
POST /api/enroll  OR  enrollInCourse() Server Action
  1. auth() → require STUDENT + ACTIVE
  2. Check course is PUBLISHED and isPaid = false
  3. Upsert Enrollment { userId, courseId }
  4. Return { ok: true, enrolled: true }
```

### 6.3 Paid Course Pipeline (Stripe)

```
Student clicks "Buy Course" on /courses/[courseId]
  │
  ├─ handlePaidCheckout() Server Action
  │   1. Check existing PAID Payment → if found, upsert Enrollment + return enrolled=true
  │   2. Apply promo code if provided:
  │       - If 100% discount: create Enrollment + PAID Payment + increment usedCount (DB txn)
  │       - Return enrolled=true (skip Stripe)
  │   3. Else: create PENDING Payment record
  │   4. POST /api/stripe/checkout → Stripe.checkout.sessions.create()
  │   5. Return { url: stripeSessionUrl }
  │
  ├─ Redirect → Stripe hosted checkout page
  │
  ├─ User completes payment → Stripe sends webhook to /api/stripe/webhook
  │   1. Verify Stripe signature (STRIPE_WEBHOOK_SECRET)
  │   2. On checkout.session.completed:
  │       - Find PENDING Payment by stripeCheckoutSessionId
  │       - Update Payment.status = PAID
  │       - Upsert Enrollment { userId, courseId }
  │   3. Return 200
  │
  └─ Redirect → /courses/[courseId]/thank-you
      - Shows ConfettiEffect (canvas-confetti)
      - Shows "Course unlocked" message

Resilience: ensureEnrollmentForPaidCoursePayment() is called on every
course page visit — if a PAID Payment exists but Enrollment is missing
(e.g. webhook delay), it silently re-creates the Enrollment.
```

### 6.4 Quiz Attempt Pipeline

```
Student opens quiz lesson → getQuizAttemptStatus() Server Action
  - Returns: { attemptsUsed, retryAt } or null if COMPLETED (exempt)

Student starts quiz → startQuizAttempt() Server Action
  1. Check UserProgress.status !== COMPLETED (passed students always exempt)
  2. Count failed QuizSubmissions in last 8 hours → max 3
  3. Create QuizAttemptSession (anti-replay token, 30min expiry)
  4. Return session token

Student submits quiz → submitQuiz() Server Action
  1. Validate session token (not expired, belongs to user+lesson)
  2. Score answers server-side (never trust client score)
  3. Create QuizSubmission + QuizSubmissionAnswer rows
  4. If passed (score >= passingScore):
      - Upsert UserProgress { status: COMPLETED, score }
      - If isFinalTest: check all modules complete → upsert Certificate
  5. If failed: increment failed attempt count
  6. Return { passed, score, nextRetryAt? }
```

### 6.5 Certificate Pipeline

```
getCourseCertificateStatus() Server Action
  1. Find all Lessons where isFinalTest = true for the course
  2. Check UserProgress.status = COMPLETED for each
  3. If all passed: call createCertificateIfEligible()
      - Upsert Certificate { userId, courseId, verificationCode: uuid(), grade }
  4. Return { eligible, completedQuizzes, requiredQuizzes, grade, verificationCode, uuid }

PDF Download: GET /courses/[courseId]/certificate/download
  - Route handler (Node runtime)
  - Generates PDF with pdfkit
  - Embeds student name, course title, grade, date, verification URL
  - Uses iconv-lite for non-Latin character encoding
  - Streams PDF to browser with Content-Disposition: attachment

QR Code: Generated client-side with `qrcode` package
  - Points to https://kabullearn.com/verify/{uuid}
  - No third-party service; works offline and in print
```

### 6.6 Video Heartbeat Pipeline

```
VideoPlayer component (client) → POST /api/lesson/heartbeat every N seconds
  - Payload: { lessonId, courseId, positionSec, consumedPct }
  - Auth required (middleware blocks unauthenticated)
  - Upserts LessonHeartbeat row
  - Updates UserProgress.status = IN_PROGRESS if not already COMPLETED
```

### 6.7 AI Course Chat Pipeline

```
Student types message in CourseChatbox
  → POST /api/chat (streaming)
  → OpenAI GPT with course context (lesson body, embeddings)
  → Streaming response via ReadableStream
  → Saves AiChatLog row (userId, courseId, message, response)
  → Thumbs up/down → POST /api/chat/feedback
```

## 7. Internationalization (i18n) System

### Languages Supported

| Code | Language | Script | Direction |
|---|---|---|---|
| `en` | English | Latin | LTR |
| `ps` | Pashto | Arabic script | RTL |
| `fa` | Dari/Persian | Arabic script | RTL |

**Scope:** All student-facing pages (public site, courses, dashboard, certificates) must support all three languages. Admin (`/admin`) and Educator (`/educator`) dashboards are English-only.

### Architecture

- **`lib/i18n.ts`** — Single source of truth. Contains three objects: `en`, `ps`, `fa`. Each is typed as `Dictionary`. TypeScript enforces key parity: if a key exists in `en`, it must exist in `ps` and `fa` or the build fails.
- **`LanguageProvider.tsx`** — Client component wrapping the full layout. Reads locale from the `Accept-Language` cookie or user preference stored in DB (`User.locale`). Sets `document.documentElement.dir` to `"rtl"` for PS/FA via `useEffect`.
- **`useLanguage()` hook** — Returns `{ t, locale, setLocale }`. `t` is the full dictionary for the current locale.
- **Server pages** read locale from a cookie and pick the right dictionary server-side (no hydration mismatch for initial render).

### Adding a New String

1. Add key to the `en` object in `lib/i18n.ts`
2. Add the same key to `ps` with Pashto translation
3. Add the same key to `fa` with Dari translation
4. TypeScript will error at build time if any key is missing from any locale

### RTL Handling

- `LanguageProvider` sets `document.documentElement.dir = "rtl"` for PS/FA
- CSS uses `html[dir="rtl"]` selectors for layout adjustments (flex direction, text-align, padding)
- **Important:** `letter-spacing` must be `0` for Arabic-script text — non-zero spacing breaks cursive letter connections. Applied via `html[dir="rtl"] .pr-eyebrow { letter-spacing: 0; }`
- Course content fields are stored trilingual in DB: `titleEn`, `titlePs`, `titleFa`, `bodyEn`, `bodyPs`, `bodyFa` on Lesson/Question/AnswerChoice

### Seeded IDs

Seeded course/module/lesson IDs use colon notation:
- Course: `"intro-physics"`
- Module: `"intro-physics:kinematics"`
- Lesson: `"intro-physics:position-velocity"`

Always use `encodeURIComponent(id)` when building href strings with these IDs.

## 8. Server Actions Pattern

All write operations live in `lib/actions/`. Every action follows this contract:

```typescript
// Return type: always ActionResult<T>
type ActionResult<T> = { ok: true; data: T } | { ok: false; error: string };

// Pattern for an authenticated educator action:
export async function createLesson(input: unknown): Promise<ActionResult<Lesson>> {
  const session = await requireEducator();           // throws → 401/403 redirect
  const data = LessonSchema.parse(input);            // throws → Zod error
  await canManageCourse(session.user.id, data.courseId); // ownership check
  const lesson = await prisma.lesson.create({ data });
  return { ok: true, data: lesson };
}
```

### Security rules enforced in every action:
1. `requireAdmin()` or `requireEducator()` — never trust client-supplied role
2. `canManageCourse(userId, courseId)` for educator-owned resources (admin bypasses)
3. Zod validation before any DB call
4. Quiz scoring is calculated server-side — client-submitted scores are never trusted
5. Stripe signature verification for webhook handler

### Action files in `lib/actions/`

| File | Responsibility |
|---|---|
| `auth-actions.ts` | `logout()` only |
| `login-actions.ts` | Credentials login |
| `register-actions.ts` | User registration + email verification |
| `enrollment-actions.ts` | Free enrollment, drop (non-paid), ensure-paid-enrollment |
| `quiz-actions.ts` | `startQuizAttempt`, `submitQuiz`, `getQuizAttemptStatus` |
| `certificate-actions.ts` | Eligibility check, certificate upsert |
| `course-actions.ts` | Course CRUD (educator) |
| `quiz-builder-actions.ts` | Module/lesson/quiz/question/choice CRUD (educator) |
| `video-actions.ts` | Heartbeat, mark lesson complete |
| `rating-actions.ts` | Course rating CRUD |
| `discussion-actions.ts` | Thread/reply CRUD |
| `message-actions.ts` | Direct message send/receive/inbox |
| `admin-message-actions.ts` | Admin broadcast + direct |
| `notification-actions.ts` | App notification read/mark |
| `bookmark-actions.ts` | Lesson bookmark toggle |
| `note-actions.ts` | Lesson notes CRUD |
| `streak-actions.ts` | Daily streak update |
| `promo-actions.ts` | Promo code apply |
| `review-checklist-actions.ts` | Admin course review checklist |
| `educator-request-actions.ts` | Educator application submit/approve/reject |
| `portal-settings-actions.ts` | Student profile settings update |
| `user-actions.ts` | Admin user management (promote, suspend) |
| `password-reset-actions.ts` | Forgot password + reset token |
| `oauth-actions.ts` | OAuth account linking |
| `site-settings-actions.ts` | Admin site-wide settings |
| `announcement-actions.ts` | Educator course announcements |

## 9. CSS Design System

### Tailwind v4 + CSS Custom Properties

Styling uses Tailwind CSS v4 alongside a CSS custom property system defined in `app/globals.css`. All semantic colors are CSS variables — no hardcoded hex values in components.

### Core CSS Variables (theming)

| Variable | Purpose |
|---|---|
| `--ink` | Primary text (near-black in light, near-white in dark) |
| `--ink-2` | Secondary text |
| `--muted` | Muted/placeholder text |
| `--muted-2` | Extra muted text |
| `--brand` | Primary brand blue (`#0057FF`) |
| `--brand-hover` | Brand hover state |
| `--brand-50` | Brand tint (10% opacity backgrounds) |
| `--surface` | Page/section background |
| `--card` | Card/panel background |
| `--border` | Border color |
| `--success` | Green (`#18825C`) |
| `--success-50` | Success tint |
| `--danger` | Red |
| `--danger-50` | Danger tint |
| `--warning` | Amber |
| `--warning-50` | Warning tint |
| `--radius` | Default border radius |
| `--radius-xl` | Large border radius |
| `--shadow` | Default card shadow |
| `--shadow-sm` | Small shadow |

Dark mode: set by `[data-theme="dark"]` on `<html>`. Variables redefine themselves — components need no dark-mode conditional classes.

### Utility Classes (design system)

| Class | Purpose |
|---|---|
| `.pr-page` | Page container (max-width, padding) |
| `.pr-panel` | Card/panel with border and background |
| `.pr-h1`, `.pr-h2`, `.pr-h3` | Heading scale |
| `.pr-copy` | Body text |
| `.pr-eyebrow` | Small uppercase label (12px, 700, letter-spacing 2.5px — 0 for RTL) |
| `.pr-btn-primary` | Primary filled button |
| `.pr-btn-ghost` | Ghost/outlined button |
| `.pr-btn-danger` | Danger button |
| `.pr-input` | Form input field |
| `.pr-label` | Form label |

### Component-specific CSS

Home page uses `kl-home-*` classes (defined in globals.css) for the hero section, feature list, and CTA layout. These are responsive and RTL-aware.

Student portal shell uses `.student-portal-shell`, `.student-portal-sidebar`, `.student-portal-content`, `.student-portal-nav-link` classes.

### Font

Custom font loaded from `font/` directory via Next.js font optimization. Applied to `:root`.

### Hydration Safety Rule

All `Date` formatting in Client Components must use explicit locale and field options — never `toLocaleString(undefined, { dateStyle, timeStyle })`. Safari's Node runtime and browser produce different outputs for undefined locale with dateStyle/timeStyle. Always use:

```typescript
date.toLocaleString("en-US", { month: "short", day: "numeric", year: "numeric", hour: "numeric", minute: "2-digit" })
```

## 10. Key Components

### Header (`components/Header.tsx` + `components/HeaderClient.tsx`)

Split into Server + Client. `Header.tsx` (async Server Component) calls `auth()` and passes `{ name, role } | null` to `HeaderClient.tsx`. Client renders role-specific nav:
- ADMIN → Admin link
- EDUCATOR → Educator portal
- STUDENT → My Courses (`/dashboard/my-courses`)
- Guest → Register + Sign in

`Header.tsx` wraps `auth()` in try/catch so a DB outage never crashes the root layout.

### CourseOverview (`components/CourseOverview.tsx`)

Main course page component. Handles:
- Free enrollment via `enrollInCourse()`
- Paid course checkout via `handlePaidCheckout()` → Stripe or 100% promo
- Shows `ConfettiEffect` on successful enrollment
- Module/lesson progress dots (trilingual `aria-label`)
- Certificate link when eligible
- `ensureEnrollmentForPaidCoursePayment()` on mount for webhook-delayed enrollments

### LessonView (`components/LessonView.tsx`)

Renders VIDEO, READING, and navigation between lessons. Features:
- YouTube embed (`VideoPlayer`) with heartbeat reporting
- Markdown rendering for READING lessons (`SimpleMarkdown`)
- Bookmark toggle (`LessonBookmark`)
- Lesson notes (`LessonNote`)
- "Save lesson" / "Saved" label uses i18n (`t.saveLesson` / `t.lessonSaved`)

### QuizView (`components/QuizView.tsx`)

Interactive quiz with:
- Server-validated attempt limits (3 failed per 8 hours)
- COMPLETED students fully exempt — no limits, no banner
- Anti-replay session token
- Server-side scoring
- Progress bar + score display on completion
- Auto-redirect to next lesson on pass

### DashboardView (`components/DashboardView.tsx`)

Student dashboard. Key feature: Drop Course button is hidden for paid courses (`isPaid` field from DB). Free courses can be dropped; paid courses grant lifetime access (no drop).

### CertificateView (`components/CertificateView.tsx`)

Certificate page with:
- Eligibility check (all `isFinalTest` lessons must be COMPLETED)
- QR code generated client-side (no third-party service)
- LinkedIn "Add to Profile" link with pre-filled fields
- PDF download button (`/certificate/download` route handler, `pdfkit`)
- Print stylesheet via `<style>` tag injection
- Certificate card uses hardcoded print-safe colors (white bg, brand blue borders) — exempt from dark mode theming

### ConfettiEffect (`components/ConfettiEffect.tsx`)

Client component, no DOM output. Fires `canvas-confetti` for 2.2 seconds on mount with brand colors `["#0057FF", "#18825C", "#F2C879", "#fff"]`. Used on:
- `/courses/[courseId]/thank-you` (Stripe payment confirmed)
- `CourseOverview` (100% promo or free enrollment inline)

### HeroSubtextRotator (`components/HeroSubtextRotator.tsx`)

Typewriter/eraser animation cycling through 6 hero subtext statements. Features:
- SSR-safe: initializes to `statements[0]` to prevent hydration mismatch
- Respects `prefers-reduced-motion`
- Accessible: `aria-live="polite"` region updates only on complete statements
- Height-anchored via invisible longest-string span (no layout shift)
- Cursor: fat block while erasing/typing, thin blinker while holding

### MessagesInbox (`components/MessagesInbox.tsx`)

Real-time-like DM inbox using polling. Student ↔ educator. Admin messages are read-only (one-way). Uses `getInbox()` and `getConversation()` Server Actions.

### CreatorDashboardView (`components/CreatorDashboardView.tsx`)

Full educator workspace: course list, analytics (enrollment counts, quiz pass rates), course editor trigger, review status display, student progress overview.

### AdminPromoCodesSection (`components/admin/AdminPromoCodesSection.tsx`)

Promo code manager. Supports PERCENT and FIXED_CENTS discount types. Uses full CSS variable theming (`var(--surface)`, `var(--card)`, `var(--border)`, `var(--success)`, `var(--danger)`, etc.) — no hardcoded dark-mode hex values.

## 11. Deployment

### kabullearn.com → Vercel

- Hosting platform: **Vercel** (zero-config Next.js)
- Triggered by git push to main branch
- Build command: `prisma migrate deploy && prisma generate && next build --webpack`
  - Note: uses `--webpack` to avoid Turbopack in production (Turbopack is dev-only)
- Environment variables set in Vercel dashboard
- Domain: `kabullearn.com`
- Serverless functions: all API routes and Server Actions run as Vercel Functions
- Edge runtime: `proxy.ts` middleware runs on Vercel Edge Network
- Cron jobs: `vercel.json` defines scheduled cron for `/api/cron/reengage`

### kabulhub.com → Cloudflare

- Hosting platform: **Cloudflare Pages** or Cloudflare static/worker
- Separate codebase / landing page at `kabulhub.com`
- DNS and CDN managed via Cloudflare dashboard

### Environment Variables

| Variable | Purpose |
|---|---|
| `DATABASE_URL` | Neon pooled connection string (for Prisma runtime) |
| `DATABASE_URL_UNPOOLED` | Neon direct connection (for migrations) |
| `NEXTAUTH_SECRET` | Auth.js JWT signing key |
| `NEXTAUTH_URL` | Base URL for Auth.js callbacks |
| `STRIPE_SECRET_KEY` | Stripe secret key |
| `STRIPE_WEBHOOK_SECRET` | Stripe webhook signature verification |
| `NEXT_PUBLIC_STRIPE_PUBLISHABLE_KEY` | Stripe public key (client) |
| `OPENAI_API_KEY` | OpenAI API key |
| `BLOB_READ_WRITE_TOKEN` | Vercel Blob storage token |
| `UPSTASH_REDIS_REST_URL` | Upstash Redis URL |
| `UPSTASH_REDIS_REST_TOKEN` | Upstash Redis token |
| `EMAIL_SERVER_*` | SMTP config for nodemailer |
| `EMAIL_FROM` | From address for transactional email |

### Development Workflow

```bash
# Always start fresh to avoid stale Turbopack chunks
rm -rf .next && npm run dev -- --port 3002

# NEVER run npm run build then npm run dev in the same session
# (production and dev chunks have incompatible module IDs)

# Before any Prisma CLI command:
set -a && source .env.local && set +a

# Schema change:
npx prisma migrate dev --name <name>

# Seed admin + sample data:
npm run db:seed
```

## 12. OLTP Entity-Relationship Diagram

```mermaid
erDiagram
  User ||--o{ Account : has
  User ||--o{ Session : has
  User ||--o{ Enrollment : enrolls
  User ||--o{ UserProgress : tracks
  User ||--o{ QuizSubmission : submits
  User ||--o{ Certificate : earns
  User ||--o{ CourseRating : rates
  User ||--o{ LessonHeartbeat : watches
  User ||--o{ QuizAttemptSession : starts
  User ||--o{ DiscussionThread : authors
  User ||--o{ DiscussionReply : writes
  User ||--o{ CourseReviewEvent : acts
  User ||--o{ Course : authors
  User ||--o{ Payment : pays
  User ||--o{ DirectMessage : sends
  User ||--o{ DirectMessage : receives
  User ||--o| CreatorProfile : owns_profile

  CreatorProfile ||--o{ CourseInstructor : listed_on
  CreatorProfile ||--o{ Course : public_author
  Course ||--o{ CourseInstructor : has

  Course ||--o{ Module : contains
  Module ||--o{ Lesson : contains
  Lesson ||--o| Quiz : may_have
  Quiz ||--o{ Question : contains
  Question ||--o{ AnswerChoice : has

  Course ||--o{ Enrollment : receives
  Course ||--o{ Payment : receives
  Course ||--o{ PromoCode : uses
  Payment }o--o| PromoCode : applied_with

  Lesson ||--o{ UserProgress : measured_by
  Lesson ||--o{ LessonHeartbeat : receives
  Lesson ||--o{ QuizAttemptSession : attempted_as
  Lesson ||--o{ QuizSubmission : submitted_for
  QuizSubmission ||--o{ QuizSubmissionAnswer : includes
  Question ||--o{ QuizSubmissionAnswer : answered
  AnswerChoice ||--o{ QuizSubmissionAnswer : selected

  Course ||--o{ Certificate : issues
  Course ||--o{ CourseRating : receives
  Course ||--o{ CourseReviewEvent : reviewed_by
  Course ||--o{ CourseReviewChecklistItem : checked_by
  Course ||--o{ DiscussionThread : discusses
  Course ||--o{ CourseAnnouncement : announces
  DiscussionThread ||--o{ DiscussionReply : has

  User ||--o{ LessonNote : writes
  User ||--o{ LessonBookmark : saves
  User ||--o| UserStreak : tracks
  User ||--o{ AiChatLog : chats
  User ||--o{ AppNotification : receives
```

Key cascade rules:
- `Course.authorId` → `onDelete: Restrict` (can't delete user who has courses)
- Most child content (Module, Lesson, Quiz, Question, AnswerChoice) → `onDelete: Cascade`
- `Payment.userId` → `onDelete: SetNull` (payment records preserved if user deleted)
- `Payment.promoCodeId` → `onDelete: SetNull` (code records preserved)

## 13. Analytics / Data Engineering

### Current State

All data lives in the OLTP PostgreSQL schema on Neon. No separate warehouse is in place. Analytics queries run directly on the operational DB (or a Neon read replica).

### OLTP Domains for Analytics

| Domain | Tables | Analytics Use |
|---|---|---|
| Identity | `User` | User cohorts, registration funnel, role distribution |
| Content | `Course`, `Module`, `Lesson` | Course catalog health, content depth |
| Enrollment | `Enrollment` | Enrollment counts, free vs paid distribution |
| Payments | `Payment`, `PromoCode` | Revenue, promo redemption rates |
| Progress | `UserProgress`, `LessonHeartbeat` | Lesson completion rates, video watch depth |
| Quizzes | `QuizSubmission` | Pass rates, attempt distributions, score histograms |
| Certificates | `Certificate` | Completion funnel (enrolled → certified) |
| Quality | `CourseRating`, `CourseReviewEvent` | Course NPS, review cycle time |
| Community | `DiscussionThread`, `DiscussionReply`, `DirectMessage` | Engagement depth |
| AI | `AiChatLog` | Chat usage, feature adoption |

### Key Metrics (SQL-ready)

```sql
-- Enrollment funnel: registered → enrolled → completed quiz → certificate
SELECT
  COUNT(DISTINCT u.id) AS registered,
  COUNT(DISTINCT e."userId") AS enrolled,
  COUNT(DISTINCT qs."userId") AS attempted_quiz,
  COUNT(DISTINCT c."userId") AS certified
FROM "User" u
LEFT JOIN "Enrollment" e ON e."userId" = u.id
LEFT JOIN "QuizSubmission" qs ON qs."userId" = u.id
LEFT JOIN "Certificate" c ON c."userId" = u.id
WHERE u.role = 'STUDENT';

-- Quiz pass rate by course
SELECT
  co."titleEn",
  COUNT(*) AS total_submissions,
  SUM(CASE WHEN qs.passed THEN 1 ELSE 0 END) AS passed,
  ROUND(100.0 * SUM(CASE WHEN qs.passed THEN 1 ELSE 0 END) / COUNT(*), 1) AS pass_rate_pct
FROM "QuizSubmission" qs
JOIN "Lesson" l ON l.id = qs."lessonId"
JOIN "Module" m ON m.id = l."moduleId"
JOIN "Course" co ON co.id = m."courseId"
GROUP BY co.id, co."titleEn"
ORDER BY pass_rate_pct;

-- Revenue (paid courses)
SELECT
  COALESCE(SUM(p."amountCents") / 100.0, 0) AS total_usd,
  COUNT(*) AS paid_transactions
FROM "Payment" p
WHERE p.status = 'PAID' AND p.purpose = 'COURSE';

-- Promo code effectiveness
SELECT
  pc.code,
  pc."discountType",
  pc."discountValue",
  pc."usedCount",
  pc."maxUses",
  ROUND(100.0 * pc."usedCount" / NULLIF(pc."maxUses", 0), 1) AS usage_pct
FROM "PromoCode" pc
ORDER BY pc."usedCount" DESC;

-- Video completion (max consumedPct per user-lesson)
SELECT
  l."titleEn",
  ROUND(AVG(max_pct), 1) AS avg_watched_pct
FROM (
  SELECT "lessonId", "userId", MAX("consumedPct") AS max_pct
  FROM "LessonHeartbeat"
  GROUP BY "lessonId", "userId"
) lh
JOIN "Lesson" l ON l.id = lh."lessonId"
GROUP BY l.id, l."titleEn"
ORDER BY avg_watched_pct DESC;
```

### Proposed Analytics Schema (future)

When query load grows, materialize into an `analytics` schema (same Neon project or read replica):

```
analytics.dim_user      ← from User
analytics.dim_course    ← from Course + CreatorProfile
analytics.dim_lesson    ← from Lesson + Module + Course
analytics.fact_enrollment       ← from Enrollment
analytics.fact_video_engagement ← aggregated from LessonHeartbeat (by user-lesson-day)
analytics.fact_quiz_submission  ← from QuizSubmission
analytics.fact_certificate      ← from Certificate
analytics.fact_payment          ← from Payment (no PII email)
```

Refresh strategy: incremental by `updatedAt` / `createdAt`, hourly for most dimensions, daily aggregation for heartbeats.

## 14. DB Resilience & Error Handling Patterns

### Server Component pages

Every Server Component wraps DB calls in try/catch:

```typescript
// Pages that can degrade (homepage, admin overview):
try {
  const data = await prisma.course.findMany({ ... });
  return <PageComponent data={data} />;
} catch {
  return <PageComponent data={[]} dbError={true} />;
  // Shows amber "Database temporarily unavailable" banner
}

// Pages that cannot render without data (course, lesson, quiz):
// throw → nearest error.tsx catches → shows error UI
```

### Header resilience

`components/Header.tsx` wraps `auth()` in try/catch so a DB outage never crashes the root layout and every page on the site.

### Enrollment resilience (paid courses)

`ensureEnrollmentForPaidCoursePayment(userId, courseId)` is called on every course page visit:
- Checks for a `PAID` Payment record even if no Enrollment row exists
- If found: silently upserts Enrollment (recovers from webhook delay or race condition)
- This means a student who paid will never be locked out, even if the webhook was delayed or failed

### Quiz anti-replay

`QuizAttemptSession` records enforce:
- Session tokens expire after 30 minutes
- One active session per user-lesson at a time
- Server validates token on submit — client cannot submit without a valid server-issued token

### Rate limiting

`Upstash Redis` is used for rate limiting sensitive endpoints (login attempts, quiz submissions, contact form). `RateLimitBucket` table in Postgres is the fallback.

### Data quality invariants

| Invariant | Enforcement |
|---|---|
| Every `QUIZ` lesson has exactly one `Quiz` | Educator UI + DB `lessonId @unique` |
| Module order is stable | `@@unique([courseId, order])` prevents duplicates |
| Enrollment is user-course unique | `@@unique([userId, courseId])` with upsert |
| One certificate per user-course | `@@unique([userId, courseId])` with upsert |
| PromoCode `usedCount` increment is transactional | DB transaction with Enrollment + Payment create |

## 15. Key Architectural Decisions & Rationale

### Why JWT instead of DB sessions?

Auth.js is configured with `strategy: "jwt"`. This means:
- `auth()` never queries the DB for session validation on every request
- Middleware runs at the edge with zero DB round-trips
- Only initial login touches the DB
- Trade-off: role/status changes don't take effect until the JWT expires (mitigated by `sessionVersion` field — incrementing it invalidates all existing tokens)

### Why two auth files?

`auth.config.ts` (edge-safe) vs `auth.ts` (Node-only) is a Next.js App Router requirement. Prisma and bcrypt use Node.js APIs incompatible with the V8 edge runtime. The split lets middleware use auth at the edge while server pages use the full auth flow.

### Why lifetime access for paid courses (no drop)?

Paid courses cannot be dropped by the student. Industry standard (Coursera, Udemy) is lifetime access once purchased. Dropping the Enrollment row wouldn't prevent re-enrollment anyway — `ensureEnrollmentForPaidCoursePayment()` would restore it on next visit. Drop button is hidden for `isPaid = true` courses; `dropEnrollment()` Server Action rejects paid courses server-side.

### Why no client-side quiz scoring?

All quiz scoring happens in `submitQuiz()` Server Action. Client submits selected answer IDs; server fetches `isCorrect` from DB and calculates score. This prevents score manipulation — the client never sees correct answers before submission.

### Why canvas-confetti over CSS animation?

`canvas-confetti` renders on an off-screen canvas overlay with no DOM footprint, doesn't affect layout, and cleans up automatically. CSS confetti would require many DOM elements and is harder to time precisely.

### Why QR code generated client-side?

Using the `qrcode` npm package on the client means:
- No third-party QR service (no external request, no privacy concern, no rate limit)
- Works offline and in print
- Fast (generated in milliseconds from the verification URL)

### Why `encodeURIComponent` on course/lesson IDs?

Seeded IDs contain colons (e.g. `"intro-physics:kinematics:quiz"`). Colons in URL path segments are technically valid but can confuse some routers and proxies. `encodeURIComponent` converts them to `%3A`, making URLs safe for all environments.

### Why Neon for PostgreSQL?

Neon is serverless PostgreSQL — scales to zero when idle, has branching for safe migrations, and integrates natively with Vercel. The `DATABASE_URL` (pooled via PgBouncer) handles serverless connection spikes; `DATABASE_URL_UNPOOLED` is used for migrations that require a direct connection.

### Why Tailwind v4 + CSS custom properties (not Tailwind theming)?

Tailwind v4's native dark mode via `[data-theme="dark"]` and CSS custom properties gives more flexibility than Tailwind's built-in `dark:` prefix — the same class works in both modes without duplication. All semantic colors are one variable reference away, making theme changes a single file edit.

## 16. Known Bugs Fixed & Patterns to Avoid

### Safari Hydration Mismatch (`toLocaleString`)

**Problem:** `date.toLocaleString(undefined, { dateStyle: "medium", timeStyle: "short" })` produces different output between Node.js (server) and Safari (client):
- Node: `Jun 11, 2026, 6:18 PM`
- Safari: `Jun 11, 2026 at 6:18 PM`

This causes a React hydration mismatch error visible in Safari DevTools.

**Fix:** Always pin the locale to `"en-US"` and use explicit individual fields (never `dateStyle`/`timeStyle`):
```typescript
// Correct — consistent across all runtimes
date.toLocaleString("en-US", { month: "short", day: "numeric", year: "numeric", hour: "numeric", minute: "2-digit" })
```

**Files fixed:** `AdminComposeForm.tsx`, `CreatorDashboardView.tsx`, `MessagesInbox.tsx`

---

### RTL Letter-Spacing Breaks Arabic Script

**Problem:** CSS `letter-spacing` greater than 0 inserts pixel gaps between glyphs. Arabic, Pashto, and Dari are cursive scripts — letters connect to neighbors. Letter-spacing visually breaks these connections, making text look like separated fragments (especially visible in Safari).

**Fix:** `html[dir="rtl"] .pr-eyebrow { letter-spacing: 0; }` in `globals.css`

**Rule:** Never apply non-zero `letter-spacing` to Arabic-script text. Applied via RTL selector, not inline.

---

### Promo Code Blocked by Existing PAID Payment

**Problem:** If a student already has a PAID Payment record for a course (e.g. bought it before), `ensureEnrollmentForPaidCoursePayment()` in the checkout route restores enrollment before promo code validation runs. The student sees "enrolled" without the promo being checked.

**Root cause:** The early-return guard for existing PAID payments runs before promo code logic.

**Fix (for testing):** Clear Payment records in DB. **For production:** the behavior is correct — a paid student shouldn't be able to use a promo code to get a "refund" on a past purchase.

---

### Turbopack Cache Corruption

**Problem:** After fixes, old compiled chunks can persist in `.next/` and still be served — making a fix appear broken even though the source is correct.

**Fix:** Always `rm -rf .next && npm run dev -- --port 3002` to start fresh.

**Never:** Run `npm run build` then `npm run dev` in the same session — production chunks have different module IDs than dev chunks.

---

### `viewFullCertificate` Duplicate Key in i18n.ts

**Problem:** Adding a new key that already exists in the `Dictionary` type causes TypeScript error "object literal may not have duplicate property names".

**Prevention:** Before adding any key to `lib/i18n.ts`, `grep` for it first: `grep -n "keyName" lib/i18n.ts`

---

### Neon MCP Multi-Statement

**Problem:** Running multiple SQL statements in a single `mcp__Neon__run_sql` call returns `NeonDbError: cannot insert multiple commands into a prepared statement`.

**Fix:** Run each statement as a separate MCP tool call. Or use `mcp__Neon__run_sql_transaction` for multi-statement transactions.

## 17. Testing

### Test framework

**Vitest** (`vitest.config.mts`) — fast Vite-based test runner, TypeScript-native.

```bash
npm test                                        # all tests
npx vitest run tests/lib/rbac.test.ts           # single file
```

Tests live in `tests/`.

### What is tested

| Area | File | Coverage |
|---|---|---|
| RBAC (`requireAdmin`, `requireEducator`) | `tests/lib/rbac.test.ts` | Role + status gate logic |

### Philosophy

Integration-focused where it matters (RBAC, quiz scoring). No mocking of Prisma in critical paths — mock divergence has caused prod failures in the past.

---

## 18. File Structure Reference

```
/
├── app/                          # Next.js App Router
│   ├── layout.tsx                # Root layout (Header, LanguageProvider, theme)
│   ├── page.tsx                  # Homepage (server, locale-aware)
│   ├── globals.css               # All CSS: variables, utilities, components
│   ├── api/                      # Route handlers (webhooks, upload, AI, cron)
│   ├── courses/[courseId]/       # Course, lesson, quiz, certificate, thank-you
│   ├── dashboard/                # Student portal (my-courses, messages, settings, certs)
│   ├── educator/                 # Educator workspace
│   └── admin/                    # Admin panel
├── components/                   # All React components (server + client)
│   ├── Header.tsx / HeaderClient.tsx
│   ├── CourseOverview.tsx        # Course page (enrollment, purchase, progress)
│   ├── LessonView.tsx            # Lesson viewer (video, reading)
│   ├── QuizView.tsx              # Interactive quiz
│   ├── CertificateView.tsx       # Certificate + PDF download
│   ├── ConfettiEffect.tsx        # canvas-confetti wrapper
│   ├── DashboardView.tsx         # Student dashboard
│   ├── CreatorDashboardView.tsx  # Educator workspace
│   ├── MessagesInbox.tsx         # DM inbox
│   ├── HeroSubtextRotator.tsx    # Homepage typewriter animation
│   ├── LanguageProvider.tsx      # i18n context + RTL
│   └── admin/                    # Admin-specific components
├── lib/
│   ├── i18n.ts                   # EN/PS/FA dictionary (Dictionary type enforces parity)
│   ├── db.ts                     # Prisma singleton
│   ├── rbac.ts                   # requireAdmin(), requireEducator(), canManageCourse()
│   ├── actions/                  # All Server Actions (26 files)
│   ├── email-verification.ts     # Email sending utilities
│   └── course-review.ts          # Review workflow helpers
├── prisma/
│   ├── schema.prisma             # Full DB schema (all models, enums, indexes)
│   ├── migrations/               # Prisma migration history
│   └── seed.mjs                  # Admin user + sample curriculum seeder
├── proxy.ts                      # Next.js middleware (edge, auth, role gates)
├── auth.config.ts                # Edge-safe Auth.js config (no Prisma/bcrypt)
├── auth.ts                       # Full Auth.js config (PrismaAdapter, bcrypt)
├── types/                        # TypeScript type extensions (next-auth augmentation)
├── tests/                        # Vitest tests
├── public/                       # Static assets (icons, images, coming-soon.html)
├── data/data.json                # Seed data (courses, modules, lessons)
└── global.ipynb                  # This documentation notebook
```